In [ ]:
%pip install pandas geopandas pillow matplotlib tqdm

In [ ]:
from pathlib import Path
import json
import pandas as pd
import geopandas as gpd
import matplotlib.pyplot as plt
from PIL import Image
from IPython.display import display

BASE_DIR = Path(
    r"C:\Users\joshu\OneDrive - National University of Singapore"
    r"\_MArch\_MArch 1\Sem 1"
    r"\AR5958A Artificial Intelligence for Urban Planning"
    r"\Group Assignment\20261106\svi_output"
)

IMAGE_DIR = BASE_DIR / "svi_images"
GEOJSON_PATH = BASE_DIR / "SVI_LeftRight_acquired.geojson"
OUTPUT_DIR = BASE_DIR / "vlm_output"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

print("Image folder exists:", IMAGE_DIR.exists())
print("GeoJSON exists:", GEOJSON_PATH.exists())
print("Output directory:", OUTPUT_DIR)

In [ ]:
gdf = gpd.read_file(GEOJSON_PATH)

# Keep only acquired records
acquired = gdf[
    gdf["image_status"] == "acquired"
].copy()

# Match relative image paths to actual files
acquired["full_image_path"] = acquired["image_path"].apply(
    lambda p: str(BASE_DIR / Path(p))
)

acquired["file_exists"] = acquired["full_image_path"].apply(
    lambda p: Path(p).is_file()
)

print("Total GeoJSON records:", len(gdf))
print("Acquired records:", len(acquired))
print("Images found:", acquired["file_exists"].sum())
print("Missing image files:", (~acquired["file_exists"]).sum())

display(acquired.head())

In [ ]:
valid = acquired[acquired["file_exists"]].copy()

sample = valid.sample(
    n=min(12, len(valid)),
    random_state=42
)

fig, axes = plt.subplots(3, 4, figsize=(16, 10))
axes = axes.flatten()

for ax, (_, row) in zip(axes, sample.iterrows()):
    with Image.open(row["full_image_path"]) as img:
        ax.imshow(img)

    ax.set_title(
        f"{row['image_id']}\n"
        f"Captured: {row['capture_date']}",
        fontsize=9
    )
    ax.axis("off")

for ax in axes[len(sample):]:
    ax.axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import subprocess

result = subprocess.run(
    [
        "nvidia-smi",
        "--query-gpu=name,memory.total,memory.free",
        "--format=csv,noheader"
    ],
    capture_output=True,
    text=True
)

if result.returncode == 0:
    print(result.stdout)
else:
    print("GPU information unavailable:")
    print(result.stderr)

In [ ]:
import requests

OLLAMA_URL = "http://localhost:11434"

try:
    response = requests.get(
        f"{OLLAMA_URL}/api/tags",
        timeout=10
    )
    response.raise_for_status()

    models = response.json().get("models", [])

    print("Ollama connected successfully.")
    print("\nAvailable models:")

    for model in models:
        print("-", model["name"])

except Exception as e:
    print("Could not connect to Ollama.")
    print("Error:", e)

In [ ]:
%pip install ollama tqdm

In [ ]:
ACCESSIBILITY_FEATURES = [
    "stairs",
    "kerb",
    "ramp",
    "lift",
    "handrail",
    "tactile_paving",
    "uneven_surface",
    "narrow_walkway",
    "construction_obstruction",
    "other_obstruction",
    "covered_walkway",
    "seating",
    "shade",
    "zebra_crossing",
    "signalised_crossing",
    "pedestrian_refuge",
    "overhead_bridge",
    "underpass"
]

print(f"Defined {len(ACCESSIBILITY_FEATURES)} features.")

In [ ]:
ACCESSIBILITY_FEATURES = [
    "sidewalk",
    "stairs",
    "kerb",
    "dropped_kerb",
    "ramp",
    "lift",
    "handrail",
    "tactile_paving",
    "uneven_surface",
    "narrow_walkway",
    "construction_obstruction",
    "other_obstruction",
    "covered_walkway",
    "seating",
    "tree_canopy",
    "zebra_crossing",
    "signalised_crossing",
    "pedestrian_refuge",
    "overhead_bridge",
    "underpass"
]

FEATURE_PROMPT = """
You are identifying pedestrian infrastructure in a
street-level photograph from Singapore.

Identify only features that are visually supported.

For every feature listed below, return one classification:
- present: clearly visible
- not_visible: not identifiable in the image
- uncertain: possibly visible but ambiguous

FEATURES:
""" + ", ".join(ACCESSIBILITY_FEATURES) + """

Important distinctions:
- A sloping road is not a pedestrian ramp.
- A normal kerb is not a dropped kerb.
- A roadside guardrail is not a pedestrian handrail.
- A building canopy is not necessarily a covered walkway.
- Road markings alone do not establish a zebra crossing.
- A vehicle is an obstruction only if it visibly blocks a pedestrian path.
- Do not infer accessibility or compliance.
- Not visible does not mean absent from the location.

Return JSON only, with all listed features:

{
  "features": {
    "sidewalk": "present",
    "stairs": "not_visible"
  }
}

No descriptions or additional commentary.
"""

In [ ]:
import json
import ollama

MODEL_NAME = "qwen2.5vl:3b"

def analyse_svi_image(image_path, model=MODEL_NAME):

    response = ollama.chat(
        model=model,
        messages=[
            {
                "role": "user",
                "content": FEATURE_PROMPT,
                "images": [str(image_path)]
            }
        ],
        format="json",
        options={
            "temperature": 0,
            "num_predict": 600
        }
    )

    result = json.loads(
        response["message"]["content"]
    )

    features = result.get("features", {})

    allowed = {"present", "not_visible", "uncertain"}

    # Standardise and validate the model output
    clean_features = {}

    for feature in ACCESSIBILITY_FEATURES:
        value = features.get(feature, "uncertain")

        if value not in allowed:
            value = "uncertain"

        clean_features[feature] = value

    return {"features": clean_features}

In [ ]:
import time
import json
from IPython.display import display

test_image = valid.iloc[0]["full_image_path"]

models = [
    "qwen2.5vl:3b",
    "qwen2.5vl:7b"
]

comparison = {}

for model in models:

    print(f"\n{'='*60}")
    print("Testing:", model)
    print("="*60)

    start = time.time()

    try:
        result = analyse_svi_image(
            test_image,
            model=model
        )

        elapsed = round(time.time() - start, 2)

        comparison[model] = {
            "result": result,
            "processing_seconds": elapsed
        }

        print("Processing time:", elapsed, "seconds")
        print(json.dumps(result, indent=2))

    except Exception as e:
        print("Error:", e)

In [ ]:
import pandas as pd

print("Available road classifications:")
print(valid["osm_class"].value_counts(dropna=False))

# Select up to 3 images from each road classification
sample_groups = []

for road_class, group in valid.groupby("osm_class"):

    selected = group.sample(
        n=min(3, len(group)),
        random_state=42
    )

    sample_groups.append(selected)

# Combine samples and randomly select 12
sample_images = pd.concat(
    sample_groups,
    ignore_index=True
)

sample_images = sample_images.sample(
    n=min(12, len(sample_images)),
    random_state=42
).reset_index(drop=True)

print("\nSelected images:", len(sample_images))

display(
    sample_images[
        [
            "image_id",
            "side",
            "osm_class",
            "capture_date"
        ]
    ]
)

In [ ]:
import time
import json
from tqdm.auto import tqdm

MODEL_NAME = "qwen2.5vl:7b"

sample_results = []

for _, row in tqdm(
    sample_images.iterrows(),
    total=len(sample_images)
):

    start = time.time()

    try:
        prediction = analyse_svi_image(
            row["full_image_path"],
            model=MODEL_NAME
        )

        sample_results.append({
            "image_id": row["image_id"],
            "image_path": row["full_image_path"],
            "osm_class": row["osm_class"],
            "prediction": prediction,
            "processing_seconds": round(
                time.time() - start, 2
            ),
            "status": "success"
        })

    except Exception as e:
        sample_results.append({
            "image_id": row["image_id"],
            "status": "error",
            "error": str(e)
        })

print("Completed:", len(sample_results))

In [ ]:
import matplotlib.pyplot as plt
from PIL import Image
import textwrap

successful_results = [
    r for r in sample_results
    if r["status"] == "success"
]

fig, axes = plt.subplots(
    len(successful_results),
    2,
    figsize=(15, 5 * len(successful_results))
)

if len(successful_results) == 1:
    axes = [axes]

for i, record in enumerate(successful_results):

    with Image.open(record["image_path"]) as img:
        axes[i][0].imshow(img)

    axes[i][0].set_title(record["image_id"])
    axes[i][0].axis("off")

    prediction = record["prediction"]

    present = [
        feature
        for feature, status
        in prediction.get("features", {}).items()
        if status == "present"
    ]

    uncertain = [
        feature
        for feature, status
        in prediction.get("features", {}).items()
        if status == "uncertain"
    ]

    summary = (
        "PRESENT:\n"
        + (", ".join(present) or "None")
        + "\n\nUNCERTAIN:\n"
        + (", ".join(uncertain) or "None")
        + "\n\nDESCRIPTION:\n"
        + prediction.get("scene_description", "")
    )

    wrapped = "\n".join(
        textwrap.fill(line, width=48)
        for line in summary.split("\n")
    )

    axes[i][1].text(
        0.02, 0.98,
        wrapped,
        va="top",
        ha="left",
        fontsize=10,
        family="monospace"
    )

    axes[i][1].axis("off")

plt.tight_layout()
plt.show()

In [ ]:
import json
import time
import pandas as pd
from pathlib import Path
from tqdm.auto import tqdm

MODEL_NAME = "qwen2.5vl:3b"

RESULTS_PATH = OUTPUT_DIR / "svi_vlm_3b_results.jsonl"

PRIORITY_CLASSES = [
    "steps",
    "pedestrian",
    "footway"
]

# Filter the existing valid GeoDataFrame
priority = valid[
    valid["osm_class"].isin(PRIORITY_CLASSES)
].copy()

# Process steps first, then pedestrian paths, then footways
priority["class_order"] = priority["osm_class"].map({
    "steps": 0,
    "pedestrian": 1,
    "footway": 2
})

priority = priority.sort_values(
    ["class_order", "image_id"]
).reset_index(drop=True)

print("Priority image distribution:")
print(priority["osm_class"].value_counts())

print("\nTotal priority images:", len(priority))

# Read previously successful results
processed_ids = set()

if RESULTS_PATH.exists():
    with open(RESULTS_PATH, "r", encoding="utf-8") as f:
        for line in f:
            try:
                record = json.loads(line)

                if (
                    record.get("status") == "success"
                    and record.get("model") == MODEL_NAME
                ):
                    processed_ids.add(record["image_id"])

            except json.JSONDecodeError:
                continue

remaining = priority[
    ~priority["image_id"].isin(processed_ids)
].copy()

print("Already completed:", len(priority) - len(remaining))
print("Remaining:", len(remaining))

# None = process every remaining priority image
# Use 30 for a short test if desired
MAX_IMAGES = None

if MAX_IMAGES is not None:
    remaining = remaining.head(MAX_IMAGES)

completed = 0
errors = 0
start_total = time.time()

with open(RESULTS_PATH, "a", encoding="utf-8") as f:

    progress = tqdm(
        remaining.iterrows(),
        total=len(remaining),
        desc="Processing pedestrian SVI",
        unit="image"
    )

    for _, row in progress:

        start = time.time()

        record = {
            "image_id": row["image_id"],
            "sample_id": row["sample_id"],
            "side": row["side"],
            "segment_id": int(row["segment_id"]),
            "latitude": float(row["latitude"]),
            "longitude": float(row["longitude"]),
            "capture_date": str(row["capture_date"]),
            "osm_class": str(row["osm_class"]),
            "image_path": str(row["image_path"]),
            "model": MODEL_NAME,
            "prompt_version": "3b_v1"
        }

        try:
            prediction = analyse_svi_image(
                row["full_image_path"],
                model=MODEL_NAME
            )

            features = prediction["features"]

            record["features"] = features
            record["status"] = "success"

            completed += 1

        except Exception as e:
            record["status"] = "error"
            record["error"] = str(e)
            errors += 1

        record["processing_seconds"] = round(
            time.time() - start, 2
        )

        f.write(json.dumps(record) + "\n")
        f.flush()

        elapsed = time.time() - start_total

        if completed + errors > 0:
            average = elapsed / (completed + errors)
            remaining_count = (
                len(remaining) - completed - errors
            )

            eta_minutes = (
                remaining_count * average / 60
            )

            progress.set_postfix({
                "success": completed,
                "errors": errors,
                "ETA_min": round(eta_minutes, 1)
            })

print("\nProcessing finished.")
print("Successful:", completed)
print("Errors:", errors)
print("Results:", RESULTS_PATH)

In [ ]:
import pandas as pd
import geopandas as gpd
import json

records = []

with open(RESULTS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError:
            continue

# Keep the latest successful record per image
successful = [
    r for r in records
    if r.get("status") == "success"
]

df = pd.DataFrame(successful)

df = df.drop_duplicates(
    subset="image_id",
    keep="last"
)

# Expand feature predictions into columns
feature_df = pd.json_normalize(
    df["features"]
)

feature_df.index = df.index

output_df = pd.concat(
    [
        df.drop(columns=["features"]),
        feature_df
    ],
    axis=1
)

# Create point geometry
output_gdf = gpd.GeoDataFrame(
    output_df,
    geometry=gpd.points_from_xy(
        output_df["longitude"],
        output_df["latitude"]
    ),
    crs="EPSG:4326"
)

# Export CSV
csv_path = OUTPUT_DIR / "svi_accessibility_3b.csv"

output_df.to_csv(
    csv_path,
    index=False
)

# Export GeoJSON
geojson_path = OUTPUT_DIR / "svi_accessibility_3b.geojson"

output_gdf.to_file(
    geojson_path,
    driver="GeoJSON"
)

print("Images exported:", len(output_gdf))
print("CSV:", csv_path)
print("GeoJSON:", geojson_path)

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt

feature_counts = (
    output_gdf[ACCESSIBILITY_FEATURES]
    .eq("present")
    .sum()
    .sort_values(ascending=True)
)

plt.figure(figsize=(10, 8))

feature_counts.plot(
    kind="barh"
)

plt.title(
    "VLM-Detected Pedestrian Features\n"
    "Tiong Bahru–Havelock"
)

plt.xlabel("Number of images classified as present")
plt.ylabel("Feature")

plt.tight_layout()
plt.show()

display(
    feature_counts.sort_values(
        ascending=False
    ).to_frame("Images classified as present")
)

In [ ]:
import json
import pandas as pd

records = []

with open(RESULTS_PATH, "r", encoding="utf-8") as f:
    for line in f:
        try:
            records.append(json.loads(line))
        except json.JSONDecodeError:
            continue

progress_df = pd.DataFrame(records)

successful = progress_df[
    (progress_df["status"] == "success")
    & (progress_df["model"] == "qwen2.5vl:3b")
].drop_duplicates(
    subset="image_id",
    keep="last"
)

priority_ids = set(priority["image_id"])

priority_completed = successful[
    successful["image_id"].isin(priority_ids)
]

print("Priority images:", len(priority))
print("Completed:", len(priority_completed))
print("Remaining:", len(priority) - len(priority_completed))

print(
    "Completion:",
    round(
        len(priority_completed) / len(priority) * 100,
        2
    ),
    "%"
)

if len(priority_completed) > 0:
    print(
        "Average seconds per image:",
        round(
            priority_completed["processing_seconds"].mean(),
            2
        )
    )

print("\nCompleted by road classification:")
print(priority_completed["osm_class"].value_counts())

In [ ]:
import pandas as pd

step_results = priority_completed[
    priority_completed["osm_class"] == "steps"
].copy()

if len(step_results) == 0:
    print("No step-class images processed yet.")

else:
    step_results["stairs_prediction"] = (
        step_results["features"].apply(
            lambda x: x.get("stairs", "missing")
        )
    )

    print("Step-class images processed:", len(step_results))

    print("\nVLM stair classifications:")
    print(
        step_results["stairs_prediction"].value_counts()
    )

    display(
        step_results[
            [
                "image_id",
                "stairs_prediction",
                "capture_date"
            ]
        ].head(20)
    )

In [ ]:
priority_output = output_gdf[
    output_gdf["osm_class"].isin(
        ["footway", "pedestrian", "steps"]
    )
].copy()

priority_geojson = (
    OUTPUT_DIR / "svi_priority_accessibility_3b.geojson"
)

priority_output.to_file(
    priority_geojson,
    driver="GeoJSON"
)

print("Exported:", len(priority_output))
print("Saved to:", priority_geojson)

In [ ]:

# ============================================================
# QWEN2.5-VL:7B vs MASK2FORMER
# CELL 01 — ENVIRONMENT AND IMAGE INVENTORY
# ============================================================

from pathlib import Path
import requests
import pandas as pd

IMAGE_DIR = Path(
    r"C:\Users\joshu\OneDrive - National University of Singapore"
    r"\_MArch\_MArch 1\Sem 1"
    r"\AR5958A Artificial Intelligence for Urban Planning"
    r"\Group Assignment\20261108\Incoming"
    r"\svi_output_block20\svi_output_block20\svi_images"
)

OUTPUT_DIR = IMAGE_DIR.parent / "Qwen_MobilityComparison"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

if not IMAGE_DIR.exists():
    raise FileNotFoundError(IMAGE_DIR)

extensions = {".jpg", ".jpeg", ".png", ".webp"}

image_files = sorted([
    p for p in IMAGE_DIR.rglob("*")
    if p.is_file() and p.suffix.lower() in extensions
])

inventory = pd.DataFrame({
    "image_id": [p.stem for p in image_files],
    "image_path": [str(p) for p in image_files]
})

if inventory["image_id"].duplicated().any():
    raise ValueError(
        "Duplicate image IDs detected. "
        "We must resolve these before comparison."
    )

print("Total images:", len(inventory))
print("Output directory:", OUTPUT_DIR)

# Check Ollama connection
response = requests.get(
    "http://localhost:11434/api/tags",
    timeout=15
)

response.raise_for_status()

models = [
    model["name"]
    for model in response.json().get("models", [])
]

print("\nAvailable Ollama models:")
print(models)

MODEL_NAME = "qwen2.5vl:7b"

if not any(
    name.split(":")[0] == "qwen2.5vl"
    and name.split(":")[-1] == "7b"
    for name in models
):
    print(
        "\nWARNING: qwen2.5vl:7b not found. "
        "Check the installed model name."
    )

display(inventory.head(10))


In [ ]:

# ============================================================
# CELL 02 — SIX-FEATURE RECOGNITION RUBRIC
# ============================================================

FEATURE_DEFINITIONS = {
    "covered_walkway": (
        "A roofed pedestrian path, covered linkway, "
        "or sheltered walking corridor."
    ),

    "construction_obstruction": (
        "Construction barriers, hoardings, temporary "
        "works, or construction materials visibly "
        "blocking or diverting pedestrian movement."
    ),

    "stairs": (
        "A visible flight of steps or stairs used "
        "to change pedestrian level. Exclude a "
        "single kerb or isolated shallow step."
    ),

    "pedestrian_overhead_bridge": (
        "An elevated pedestrian bridge crossing "
        "above a road or other transport corridor."
    ),

    "ramp": (
        "A visibly sloped pedestrian access route "
        "connecting different ground levels. "
        "Exclude ordinary sloping roads."
    ),

    "underpass": (
        "A pedestrian passage below a road or "
        "transport corridor. Exclude ordinary "
        "covered walkways and building corridors."
    )
}

FEATURE_KEYS = list(FEATURE_DEFINITIONS.keys())

print("Recognition categories:", len(FEATURE_KEYS))

for key, definition in FEATURE_DEFINITIONS.items():
    print(f"\n{key}: {definition}")


In [ ]:

# ============================================================
# CELL 03 — QWEN 7B PILOT RECOGNITION
# 20 IMAGES | SIX MOBILITY FEATURES
# ============================================================

import base64
import json
import re
import time
from io import BytesIO
from PIL import Image, ImageOps
from tqdm.auto import tqdm

MODEL_NAME = "qwen2.5vl:7b"
PILOT_SIZE = 20

pilot = inventory.sample(
    n=min(PILOT_SIZE, len(inventory)),
    random_state=42
).reset_index(drop=True)

definitions_text = "\n".join(
    f"- {key}: {definition}"
    for key, definition in FEATURE_DEFINITIONS.items()
)

PROMPT = f"""
You are analysing street-level photographs of pedestrian
environments in Singapore.

Identify whether each of the following six mobility
features is visibly present in the image.

FEATURE DEFINITIONS:
{definitions_text}

Return ONLY a JSON object with exactly these six keys:
{json.dumps(FEATURE_KEYS)}

For each key, use:
1 = visibly present
0 = visibly absent
null = cannot determine from the image

Rules:
- Use only directly visible evidence.
- Do not infer objects outside the photograph.
- Do not classify ordinary roads as pedestrian ramps.
- Do not classify ordinary roofed corridors as underpasses.
- Do not classify single kerbs as stairs.
- Do not assume construction obstructs pedestrians
  unless an obstruction or diversion is visible.
- Multiple features may be present.
- Do not include explanations or Markdown.
"""

def encode_image(path, max_dimension=1280):

    with Image.open(path) as img:
        image = ImageOps.exif_transpose(img).convert("RGB")

    image.thumbnail((max_dimension, max_dimension))

    buffer = BytesIO()
    image.save(buffer, format="JPEG", quality=85)

    return base64.b64encode(
        buffer.getvalue()
    ).decode("utf-8")


def parse_response(text):

    text = text.strip()

    # Remove Markdown fences if present
    text = re.sub(
        r"^```(?:json)?\s*|\s*```$",
        "",
        text,
        flags=re.IGNORECASE
    ).strip()

    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if not match:
            raise ValueError("No JSON object returned.")
        data = json.loads(match.group(0))

    if not isinstance(data, dict):
        raise ValueError("Response is not a JSON object.")

    if set(data.keys()) != set(FEATURE_KEYS):
        raise ValueError(
            "Missing or unexpected feature keys."
        )

    for key in FEATURE_KEYS:
        if data[key] not in (0, 1, None) or isinstance(
            data[key], bool
        ):
            raise ValueError(
                f"Invalid value for {key}: {data[key]}"
            )

    return data


def score_image(image_path):

    encoded = encode_image(image_path)

    payload = {
        "model": MODEL_NAME,
        "stream": False,
        "messages": [
            {
                "role": "user",
                "content": PROMPT,
                "images": [encoded]
            }
        ],
        "options": {
            "temperature": 0,
            "num_ctx": 4096
        }
    }

    response = requests.post(
        "http://localhost:11434/api/chat",
        json=payload,
        timeout=300
    )

    response.raise_for_status()

    text = response.json()["message"]["content"]

    return parse_response(text), text


records = []

for _, row in tqdm(
    pilot.iterrows(),
    total=len(pilot),
    desc="Qwen mobility recognition"
):

    start = time.time()

    record = {
        "image_id": row["image_id"],
        "image_path": row["image_path"]
    }

    try:
        scores, raw = score_image(row["image_path"])

        record.update(scores)
        record["status"] = "success"
        record["raw_response"] = raw

    except Exception as e:
        record.update({
            key: None for key in FEATURE_KEYS
        })
        record["status"] = "failed"
        record["raw_response"] = ""
        record["error"] = str(e)

    record["processing_seconds"] = round(
        time.time() - start, 2
    )

    records.append(record)

    # Save progress after every image
    pd.DataFrame(records).to_csv(
        OUTPUT_DIR / "Qwen7B_Pilot20.csv",
        index=False,
        encoding="utf-8-sig"
    )

results = pd.DataFrame(records)

print("\nPILOT REPORT")
print("Images processed:", len(results))
print(
    "Successful:",
    (results["status"] == "success").sum()
)
print(
    "Failed:",
    (results["status"] == "failed").sum()
)

print("\nPositive detections by category:")

display(
    results[FEATURE_KEYS]
    .eq(1)
    .sum()
    .to_frame("positive_images")
)

display(
    results[
        ["image_id", "status"] + FEATURE_KEYS
    ]
)


In [ ]:

# ============================================================
# CELL 04A — FIXED BINARY QWEN RECOGNITION
# ============================================================

import json
import re
import requests

BINARY_PROMPT = f"""
Analyse this street-level photograph from Singapore.

For each mobility feature, determine whether it is
visibly present.

FEATURE DEFINITIONS:

{definitions_text}

Return exactly six binary values in JSON:

{json.dumps({key: 0 for key in FEATURE_KEYS}, indent=2)}

SCORING RULES:
1 = visibly present
0 = not visibly identifiable

IMPORTANT:
- Every value must be 0 or 1.
- Never return null, None, unknown, or text.
- Do not infer features outside the photograph.
- A single kerb is not stairs.
- An ordinary sloping road is not a pedestrian ramp.
- A covered walkway is not an underpass.
- Construction must visibly obstruct or divert
  pedestrian movement to count as an obstruction.
- Multiple features may be present.
- Return JSON only.
"""

def parse_binary_response(text):

    text = text.strip()

    text = re.sub(
        r"^```(?:json)?\s*|\s*```$",
        "",
        text,
        flags=re.IGNORECASE
    ).strip()

    try:
        data = json.loads(text)
    except json.JSONDecodeError:
        match = re.search(r"\{.*\}", text, re.DOTALL)
        if not match:
            raise ValueError("No JSON object returned.")
        data = json.loads(match.group(0))

    if set(data.keys()) != set(FEATURE_KEYS):
        raise ValueError("Incorrect feature keys.")

    for key in FEATURE_KEYS:
        value = data[key]

        if type(value) is not int or value not in (0, 1):
            raise ValueError(
                f"Invalid value for {key}: {value}"
            )

    return data


def score_image_binary(image_path):

    encoded = encode_image(image_path)

    payload = {
        "model": "qwen2.5vl:7b",
        "stream": False,
        "messages": [
            {
                "role": "user",
                "content": BINARY_PROMPT,
                "images": [encoded]
            }
        ],
        "options": {
            "temperature": 0,
            "num_ctx": 4096
        }
    }

    response = requests.post(
        "http://localhost:11434/api/chat",
        json=payload,
        timeout=300
    )

    response.raise_for_status()

    raw = response.json()["message"]["content"]

    return parse_binary_response(raw), raw

print("Binary scoring function ready.")


In [ ]:

# ============================================================
# CELL 04B — VERIFY BINARY OUTPUT
# ============================================================

import pandas as pd
from tqdm.auto import tqdm
import time

binary_records = []

for _, row in tqdm(
    pilot.iterrows(),
    total=len(pilot),
    desc="Qwen binary recognition"
):

    start = time.time()

    record = {
        "image_id": row["image_id"],
        "image_path": row["image_path"]
    }

    try:
        scores, raw = score_image_binary(row["image_path"])

        record.update(scores)
        record["status"] = "success"
        record["raw_response"] = raw

    except Exception as e:
        record.update({
            key: None for key in FEATURE_KEYS
        })
        record["status"] = "failed"
        record["error"] = str(e)

    record["processing_seconds"] = round(
        time.time() - start, 2
    )

    binary_records.append(record)

binary_results = pd.DataFrame(binary_records)

print("\nBINARY PILOT REPORT")
print("Successful:", (binary_results.status == "success").sum())
print("Failed:", (binary_results.status == "failed").sum())

print("\nMissing values by category:")
display(binary_results[FEATURE_KEYS].isna().sum())

print("\nPositive detections:")
display(
    binary_results[FEATURE_KEYS]
    .eq(1)
    .sum()
    .to_frame("positive_images")
)

display(
    binary_results[
        ["image_id", "status"] + FEATURE_KEYS
    ]
)

binary_results.to_csv(
    OUTPUT_DIR / "Qwen7B_Pilot20_Binary.csv",
    index=False,
    encoding="utf-8-sig"
)


In [ ]:

# ============================================================
# CELL 05A — FIX WINDOWS PATH LENGTH ERROR
# ============================================================

from pathlib import Path

# Keep original image locations unchanged
IMAGE_DIR = Path(
    r"C:\Users\joshu\OneDrive - National University of Singapore"
    r"\_MArch\_MArch 1\Sem 1"
    r"\AR5958A Artificial Intelligence for Urban Planning"
    r"\Group Assignment\20261108\Incoming"
    r"\svi_output_block20\svi_output_block20\svi_images"
)

# Use a short directory for Qwen outputs
OUTPUT_DIR = Path(r"C:\Qwen_TB")

RUN_DIR = OUTPUT_DIR / "FullBatch"
RECORD_DIR = RUN_DIR / "Records"

for folder in [OUTPUT_DIR, RUN_DIR, RECORD_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Image directory:", IMAGE_DIR)
print("Output directory:", OUTPUT_DIR)
print("Record directory:", RECORD_DIR)

print("\nImage directory exists:", IMAGE_DIR.exists())
print("Output directory exists:", OUTPUT_DIR.exists())
print("\nPath length:", len(str(RECORD_DIR)))


In [ ]:

# ============================================================
# QWEN2.5-VL:7B vs MASK2FORMER
# CELL 05 — FULL-BATCH MOBILITY RECOGNITION
# Six features | Checkpointing | Resume support
# ============================================================

from pathlib import Path
from tqdm.auto import tqdm
from datetime import datetime, timezone
import pandas as pd
import numpy as np
import json
import time

# ------------------------------------------------------------
# 01 — VERIFY ENVIRONMENT
# ------------------------------------------------------------

required_vars = [
    "inventory",
    "OUTPUT_DIR",
    "FEATURE_KEYS",
    "score_image_binary"
]

missing_vars = [
    name for name in required_vars
    if name not in globals()
]

if missing_vars:
    raise RuntimeError(
        f"Missing variables: {missing_vars}. "
        "Run Cells 01, 02 and 04A first."
    )

OUTPUT_DIR = Path(OUTPUT_DIR)

RUN_DIR = OUTPUT_DIR / "FullBatch"
RECORD_DIR = RUN_DIR / "Records"

RECORD_DIR.mkdir(parents=True, exist_ok=True)

MASTER_CSV = RUN_DIR / "Qwen7B_FullBatch_Results.csv"
FAILURE_CSV = RUN_DIR / "Qwen7B_FullBatch_Failures.csv"

MAX_RETRIES = 3

# ------------------------------------------------------------
# 02 — PREPARE IMAGE INVENTORY
# ------------------------------------------------------------

full_inventory = (
    inventory.drop_duplicates("image_id")
    .sort_values("image_id")
    .reset_index(drop=True)
)

if full_inventory["image_id"].duplicated().any():
    raise ValueError("Duplicate image IDs found.")

print("=" * 65)
print("QWEN2.5-VL:7B — FULL-BATCH MOBILITY RECOGNITION")
print("=" * 65)

print("Model: qwen2.5vl:7b")
print("Images:", len(full_inventory))
print("Features:", len(FEATURE_KEYS))
print("Output:", RUN_DIR)

# ------------------------------------------------------------
# 03 — SAVE INDIVIDUAL RECORD
# ------------------------------------------------------------

def save_record(record, path):

    temporary = path.with_suffix(".tmp")

    with open(temporary, "w", encoding="utf-8") as f:
        json.dump(record, f, indent=2)

    temporary.replace(path)


# ------------------------------------------------------------
# 04 — PROCESS IMAGES
# ------------------------------------------------------------

new_success = 0
skipped = 0
failed = 0

for _, row in tqdm(
    full_inventory.iterrows(),
    total=len(full_inventory),
    desc="Qwen full-batch recognition"
):

    image_id = str(row["image_id"])
    image_path = str(row["image_path"])

    record_path = RECORD_DIR / f"{image_id}.json"

    # Skip successfully completed images
    if record_path.exists():

        try:
            with open(record_path, "r", encoding="utf-8") as f:
                previous = json.load(f)

            if (
                previous.get("status") == "success"
                and all(
                    type(previous.get(key)) is int
                    and previous[key] in (0, 1)
                    for key in FEATURE_KEYS
                )
            ):
                skipped += 1
                continue

        except Exception:
            pass

    start = time.time()

    record = {
        "image_id": image_id,
        "image_path": image_path,
        "model": "qwen2.5vl:7b",
        "protocol": "binary_v1",
        "status": "failed"
    }

    last_error = ""

    for attempt in range(1, MAX_RETRIES + 1):

        try:

            scores, raw = score_image_binary(image_path)

            record.update(scores)

            record["status"] = "success"
            record["raw_response"] = raw
            record["attempts"] = attempt

            break

        except Exception as e:

            last_error = str(e)

            if attempt < MAX_RETRIES:
                time.sleep(2 * attempt)

    if record["status"] == "success":
        new_success += 1
    else:
        failed += 1

        record.update({
            key: None for key in FEATURE_KEYS
        })

        record["error"] = last_error
        record["attempts"] = MAX_RETRIES

        print("FAILED:", image_id, last_error)

    record["processing_seconds"] = round(
        time.time() - start, 2
    )

    record["processed_at_utc"] = (
        datetime.now(timezone.utc).isoformat()
    )

    # Save immediately after each image
    save_record(record, record_path)

# ------------------------------------------------------------
# 05 — CONSOLIDATE ALL SAVED RECORDS
# ------------------------------------------------------------

records = []

for _, row in full_inventory.iterrows():

    record_path = (
        RECORD_DIR / f"{row['image_id']}.json"
    )

    if not record_path.exists():
        continue

    try:
        with open(record_path, "r", encoding="utf-8") as f:
            records.append(json.load(f))

    except Exception as e:
        print("INVALID RECORD:", record_path.name, e)

results_full = pd.DataFrame(records)

# Explicit nullable integer format
for key in FEATURE_KEYS:

    if key in results_full.columns:
        results_full[key] = pd.to_numeric(
            results_full[key],
            errors="coerce"
        ).astype("Int64")

results_full.to_csv(
    MASTER_CSV,
    index=False,
    encoding="utf-8-sig"
)

failures = results_full[
    results_full["status"] != "success"
].copy()

failures.to_csv(
    FAILURE_CSV,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 06 — SUMMARY
# ------------------------------------------------------------

successful = results_full[
    results_full["status"] == "success"
].copy()

print("\n" + "=" * 65)
print("QWEN FULL-BATCH RECOGNITION REPORT")
print("=" * 65)

print("Expected images:", len(full_inventory))
print("Saved records:", len(results_full))
print("Successful:", len(successful))
print("Failed:", len(failures))
print("Previously completed:", skipped)
print("Newly successful:", new_success)

print("\nPOSITIVE DETECTIONS BY CATEGORY")

summary = pd.DataFrame({
    "feature": FEATURE_KEYS,
    "positive_images": [
        int(successful[key].eq(1).sum())
        for key in FEATURE_KEYS
    ],
    "negative_images": [
        int(successful[key].eq(0).sum())
        for key in FEATURE_KEYS
    ]
})

display(summary)

summary.to_csv(
    RUN_DIR / "Qwen7B_FeatureSummary.csv",
    index=False,
    encoding="utf-8-sig"
)

if len(successful) > 0:

    print(
        "\nAverage seconds per image:",
        round(
            successful["processing_seconds"].mean(), 2
        )
    )

print("\nMaster results:", MASTER_CSV)
print("Failure report:", FAILURE_CSV)
print("\nCELL 05 COMPLETED")


In [ ]:
# ============================================================
# CELL 06A — LOCATE QWEN RESULTS
# ============================================================

from pathlib import Path
import pandas as pd

SEARCH_ROOTS = [
    Path(r"C:\Qwen_TB"),
    Path.home() / "Documents",
    Path.home() / "OneDrive - National University of Singapore"
]

found = []

for root in SEARCH_ROOTS:
    if root.exists():
        found.extend(
            root.rglob("Qwen7B_FullBatch_Results.csv")
        )

found = sorted(set(found))

print("Results files found:", len(found))

for i, path in enumerate(found):
    print(f"{i}: {path}")

if len(found) == 1:
    RESULTS_PATH = found[0]
    df = pd.read_csv(RESULTS_PATH)

    print("\nResults loaded successfully!")
    print("Images:", len(df))
    print("Path:", RESULTS_PATH)

elif len(found) == 0:
    print("\nNo results CSV found.")
    print("Checking current notebook variables...")

    if "results_full" in globals():
        print("Found results_full in memory.")
        print("Rows:", len(results_full))

else:
    print("\nMultiple files found. Select the correct one.")

In [ ]:
from pathlib import Path

PROJECT_DIR = Path(
    r"C:\Users\joshu\OneDrive - National University of Singapore"
    r"\_MArch\_MArch 1\Sem 1"
    r"\AR5958A Artificial Intelligence for Urban Planning"
    r"\Group Assignment\20261108\Qwen_TB"
)

RESULTS_PATH = (
    PROJECT_DIR / "FullBatch"
    / "Qwen7B_FullBatch_Results.csv"
)

# Future visualisation outputs
OUTPUT_DIR = PROJECT_DIR / "Visualisations"
ATLAS_DIR = OUTPUT_DIR / "FeatureAtlas"
JOURNEY_DIR = OUTPUT_DIR / "JourneyAnalysis"

for folder in [OUTPUT_DIR, ATLAS_DIR, JOURNEY_DIR]:
    folder.mkdir(parents=True, exist_ok=True)

print("Project directory:", PROJECT_DIR)
print("Results found:", RESULTS_PATH.exists())

In [ ]:

# ============================================================
# CELL 06 — QWEN2.5-VL:7B FEATURE DETECTION ATLAS
# FULLY REVISED | SIX MOBILITY CATEGORIES
# ============================================================

from pathlib import Path
from PIL import Image, ImageOps, ImageDraw, ImageFont
import pandas as pd
import math

# ------------------------------------------------------------
# 01 — PROJECT DIRECTORIES
# ------------------------------------------------------------

PROJECT_DIR = Path(
    r"C:\Users\joshu\OneDrive - National University of Singapore"
    r"\_MArch\_MArch 1\Sem 1"
    r"\AR5958A Artificial Intelligence for Urban Planning"
    r"\Group Assignment\20261108\Qwen_TB"
)

RESULTS_PATH = (
    PROJECT_DIR / "FullBatch"
    / "Qwen7B_FullBatch_Results.csv"
)

ATLAS_DIR = PROJECT_DIR / "Visualisations" / "FeatureAtlas"
ATLAS_DIR.mkdir(parents=True, exist_ok=True)

if not RESULTS_PATH.is_file():
    raise FileNotFoundError(RESULTS_PATH)

df = pd.read_csv(RESULTS_PATH)

print("=" * 65)
print("QWEN2.5-VL:7B — FEATURE DETECTION ATLAS")
print("=" * 65)
print("Total image records:", len(df))
print("Output directory:", ATLAS_DIR)

# ------------------------------------------------------------
# 02 — FEATURE CATEGORIES
# ------------------------------------------------------------

FEATURES = {
    "covered_walkway": "Covered Walkway",
    "construction_obstruction": "Construction Obstruction",
    "stairs": "Stairs",
    "pedestrian_overhead_bridge": "Pedestrian Overhead Bridge",
    "ramp": "Ramp",
    "underpass": "Underpass"
}

# ------------------------------------------------------------
# 03 — CONTACT SHEET SETTINGS
# ------------------------------------------------------------

IMAGES_PER_SHEET = 24
COLUMNS = 6

TILE_W = 320
TILE_H = 275
IMAGE_W = 300
IMAGE_H = 220

HEADER_H = 130
MARGIN = 12

# Optional font for Windows
font_path = Path(r"C:\Windows\Fonts\arial.ttf")
bold_path = Path(r"C:\Windows\Fonts\arialbd.ttf")

if font_path.exists():
    font = ImageFont.truetype(str(font_path), 19)
    small_font = ImageFont.truetype(str(font_path), 15)
else:
    font = ImageFont.load_default()
    small_font = ImageFont.load_default()

if bold_path.exists():
    title_font = ImageFont.truetype(str(bold_path), 34)
else:
    title_font = font

# ------------------------------------------------------------
# 04 — IMAGE LOADING HELPER
# ------------------------------------------------------------

def load_atlas_image(path):

    path = Path(str(path))

    if not path.is_file():
        return None

    try:
        with Image.open(path) as im:
            image = ImageOps.exif_transpose(im).convert("RGB")
        return image

    except Exception as e:
        print("Image loading failed:", path.name, e)
        return None


# ------------------------------------------------------------
# 05 — GENERATE SIX CONTACT SHEETS
# ------------------------------------------------------------

atlas_records = []
summary_records = []

for feature, title in FEATURES.items():

    if feature not in df.columns:
        print("Missing feature column:", feature)
        continue

    subset = df[
        (df["status"] == "success") &
        (
            pd.to_numeric(
                df[feature],
                errors="coerce"
            ) == 1
        )
    ].copy()

    total_positive = len(subset)

    # Use a fixed seed for reproducibility
    if total_positive > IMAGES_PER_SHEET:
        subset = subset.sample(
            n=IMAGES_PER_SHEET,
            random_state=42
        )

    subset = subset.sort_values(
        "image_id"
    ).reset_index(drop=True)

    rows = max(
        1,
        math.ceil(len(subset) / COLUMNS)
    )

    canvas_w = COLUMNS * TILE_W
    canvas_h = HEADER_H + rows * TILE_H

    sheet = Image.new(
        "RGB",
        (canvas_w, canvas_h),
        "white"
    )

    draw = ImageDraw.Draw(sheet)

    # --------------------------------------------------------
    # HEADER
    # --------------------------------------------------------

    draw.text(
        (30, 22),
        f"QWEN2.5-VL:7B | {title.upper()}",
        fill="#202020",
        font=title_font
    )

    draw.text(
        (30, 76),
        (
            f"Positive detections: {total_positive}  |  "
            f"Displayed: {len(subset)}  |  "
            f"Total corpus: {len(df)} images"
        ),
        fill="#666666",
        font=font
    )

    draw.line(
        (30, 112, canvas_w - 30, 112),
        fill="#CCCCCC",
        width=2
    )

    # --------------------------------------------------------
    # IMAGE GRID
    # --------------------------------------------------------

    displayed = 0

    for i, (_, record) in enumerate(
        subset.iterrows()
    ):

        col = i % COLUMNS
        row = i // COLUMNS

        x = col * TILE_W
        y = HEADER_H + row * TILE_H

        image_path = Path(
            str(record["image_path"])
        )

        image = load_atlas_image(image_path)

        if image is None:
            draw.text(
                (x + 20, y + 80),
                "IMAGE UNAVAILABLE",
                fill="#BB3333",
                font=small_font
            )
            continue

        # Preserve image aspect ratio
        image.thumbnail(
            (IMAGE_W, IMAGE_H),
            Image.Resampling.LANCZOS
        )

        px = x + (TILE_W - image.width) // 2
        py = y + 8 + (IMAGE_H - image.height) // 2

        sheet.paste(
            image,
            (px, py)
        )

        # Image ID
        draw.text(
            (x + 12, y + IMAGE_H + 20),
            str(record["image_id"]),
            fill="#222222",
            font=font
        )

        # Thin divider
        draw.line(
            (
                x + MARGIN,
                y + TILE_H - 8,
                x + TILE_W - MARGIN,
                y + TILE_H - 8
            ),
            fill="#DDDDDD",
            width=1
        )

        displayed += 1

        atlas_records.append({
            "feature": feature,
            "image_id": record["image_id"],
            "image_path": str(image_path),
            "atlas_position": i + 1
        })

    # --------------------------------------------------------
    # SAVE PNG AND JPG
    # --------------------------------------------------------

    png_path = (
        ATLAS_DIR / f"{feature}_atlas.png"
    )

    jpg_path = (
        ATLAS_DIR / f"{feature}_atlas.jpg"
    )

    sheet.save(
        png_path,
        format="PNG"
    )

    sheet.save(
        jpg_path,
        format="JPEG",
        quality=95,
        subsampling=0
    )

    summary_records.append({
        "feature": feature,
        "positive_images": total_positive,
        "displayed_images": displayed,
        "png_path": str(png_path),
        "jpg_path": str(jpg_path)
    })

    print(
        f"{title}: "
        f"{total_positive} positive | "
        f"{displayed} displayed"
    )

# ------------------------------------------------------------
# 06 — EXPORT ATLAS INVENTORY
# ------------------------------------------------------------

atlas_inventory = pd.DataFrame(
    atlas_records
)

atlas_summary = pd.DataFrame(
    summary_records
)

atlas_inventory.to_csv(
    ATLAS_DIR / "FeatureAtlas_ImageInventory.csv",
    index=False,
    encoding="utf-8-sig"
)

atlas_summary.to_csv(
    ATLAS_DIR / "FeatureAtlas_Summary.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 07 — FINAL REPORT
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("FEATURE DETECTION ATLAS — COMPLETE")
print("=" * 65)

display(
    atlas_summary[
        [
            "feature",
            "positive_images",
            "displayed_images"
        ]
    ]
)

print("\nOutput directory:")
print(ATLAS_DIR)

print("\nSix contact sheets generated.")
print("PNG + JPG versions saved.")


In [ ]:

# ============================================================
# CELL 08 — QWEN MOBILITY INTELLIGENCE
# COMPLETE UI HANDOVER PACKAGE
# ============================================================

from pathlib import Path
from datetime import datetime, timezone
from collections import Counter
import pandas as pd
import geopandas as gpd
import json
import shutil
import re

# ------------------------------------------------------------
# 01 — PROJECT PATHS
# ------------------------------------------------------------

PROJECT_DIR = Path(
    r"C:\Users\joshu\OneDrive - National University of Singapore"
    r"\_MArch\_MArch 1\Sem 1"
    r"\AR5958A Artificial Intelligence for Urban Planning"
    r"\Group Assignment\20261108\Qwen_TB"
)

RESULTS_PATH = (
    PROJECT_DIR / "FullBatch"
    / "Qwen7B_FullBatch_Results.csv"
)

HANDOVER_DIR = PROJECT_DIR / "UI_Handover"

IMAGE_OUT = HANDOVER_DIR / "images"
DATA_OUT = HANDOVER_DIR / "data"
DOC_OUT = HANDOVER_DIR / "documentation"
VALIDATION_OUT = HANDOVER_DIR / "validation"

for folder in [
    IMAGE_OUT, DATA_OUT, DOC_OUT, VALIDATION_OUT
]:
    folder.mkdir(parents=True, exist_ok=True)

if not RESULTS_PATH.is_file():
    raise FileNotFoundError(RESULTS_PATH)

df = pd.read_csv(RESULTS_PATH)

# ------------------------------------------------------------
# 02 — FEATURE DEFINITIONS
# ------------------------------------------------------------

FEATURES = [
    "covered_walkway",
    "construction_obstruction",
    "stairs",
    "pedestrian_overhead_bridge",
    "ramp",
    "underpass"
]

FEATURE_LABELS = {
    "covered_walkway": "Covered walkway",
    "construction_obstruction": "Construction obstruction",
    "stairs": "Stairs",
    "pedestrian_overhead_bridge": "Pedestrian overhead bridge",
    "ramp": "Ramp",
    "underpass": "Underpass"
}

for feature in FEATURES:
    df[feature] = pd.to_numeric(
        df[feature], errors="coerce"
    ).astype("Int64")

df = df[df["status"] == "success"].copy()

# ------------------------------------------------------------
# 03 — IDENTIFY SAMPLING POINTS
# ------------------------------------------------------------

parts = df["image_id"].str.extract(
    r"^(?P<sample_id>.+)_(?P<side>[LR])$"
)

df["sample_id"] = parts["sample_id"]
df["side"] = parts["side"]

# ------------------------------------------------------------
# 04 — FIND ORIGINAL GEOGRAPHIC METADATA
# ------------------------------------------------------------

SEARCH_ROOT = PROJECT_DIR.parent / "Incoming"

candidate_files = []

if SEARCH_ROOT.exists():
    for extension in ["*.geojson", "*.csv", "*.gpkg"]:
        candidate_files.extend(
            SEARCH_ROOT.rglob(extension)
        )

print("Potential geographic metadata files:",
      len(candidate_files))

# Use only metadata with explicit image/sample IDs.
# Avoid guessing coordinates from row order.

geo_candidates = []

for path in candidate_files:

    try:
        if path.suffix.lower() == ".csv":
            metadata = pd.read_csv(path, nrows=5)
        elif path.suffix.lower() == ".geojson":
            metadata = gpd.read_file(path, rows=5)
        else:
            continue

        columns = {
            str(c).lower() for c in metadata.columns
        }

        has_id = bool(
            columns & {
                "image_id", "sample_id", "point_id"
            }
        )

        has_location = (
            {"latitude", "longitude"}.issubset(columns)
            or "geometry" in columns
        )

        if has_id and has_location:
            geo_candidates.append(path)

    except Exception:
        continue

print("\nGeographic metadata candidates:")

for i, path in enumerate(geo_candidates):
    print(f"{i}: {path}")

# ------------------------------------------------------------
# 05 — SAFE GEOGRAPHIC MATCHING
# ------------------------------------------------------------

df["latitude"] = float("nan")
df["longitude"] = float("nan")
df["geolocation_status"] = "unmatched"

def normalise_id(value):
    return str(value).strip().upper()

def read_geographic_file(path):

    if path.suffix.lower() == ".csv":

        meta = pd.read_csv(path)

        if {"latitude", "longitude"}.issubset(meta.columns):
            return meta

        return None

    geo = gpd.read_file(path)

    if geo.crs is None:
        return None

    geo = geo.to_crs("EPSG:4326")

    geo["latitude"] = geo.geometry.y
    geo["longitude"] = geo.geometry.x

    return pd.DataFrame(
        geo.drop(columns="geometry")
    )

if len(geo_candidates) == 1:

    metadata = read_geographic_file(
        geo_candidates[0]
    )

    if metadata is not None:

        id_column = next(
            (
                col for col in [
                    "image_id", "sample_id", "point_id"
                ]
                if col in metadata.columns
            ),
            None
        )

        if id_column:

            metadata["join_id"] = (
                metadata[id_column]
                .map(normalise_id)
            )

            # Only use unambiguous coordinate assignments
            metadata = metadata.drop_duplicates(
                "join_id", keep=False
            )

            df["join_id"] = (
                df["image_id"].map(normalise_id)
                if id_column == "image_id"
                else df["sample_id"].map(normalise_id)
            )

            lookup = metadata.set_index("join_id")

            df["latitude"] = df["join_id"].map(
                lookup["latitude"]
            )

            df["longitude"] = df["join_id"].map(
                lookup["longitude"]
            )

            matched_geo = (
                df["latitude"].notna()
                & df["longitude"].notna()
            )

            df.loc[
                matched_geo,
                "geolocation_status"
            ] = "matched"

# ------------------------------------------------------------
# 06 — COPY SOURCE IMAGES
# ------------------------------------------------------------

copy_records = []

for _, row in df.iterrows():

    image_id = str(row["image_id"])
    source = Path(str(row["image_path"]))

    destination = (
        IMAGE_OUT / f"{image_id}{source.suffix.lower()}"
    )

    status = "missing"

    if source.is_file():

        if not destination.exists():
            shutil.copy2(source, destination)

        status = "available"

    copy_records.append({
        "image_id": image_id,
        "image_status": status,
        "relative_image_path": (
            f"images/{destination.name}"
            if status == "available"
            else ""
        )
    })

copies = pd.DataFrame(copy_records)

df = df.merge(
    copies,
    on="image_id",
    how="left"
)

# ------------------------------------------------------------
# 07 — CLASSIFY ATTENTION STATUS
# ------------------------------------------------------------

BARRIERS = [
    "construction_obstruction",
    "stairs"
]

CONDITIONAL = [
    "pedestrian_overhead_bridge",
    "underpass"
]

SUPPORTIVE = [
    "covered_walkway",
    "ramp"
]

def classify_attention(row):

    if any(row[k] == 1 for k in BARRIERS):
        return "POTENTIAL_BARRIER"

    if any(row[k] == 1 for k in CONDITIONAL):
        return "VERIFY_STEP_FREE_ACCESS"

    if any(row[k] == 1 for k in SUPPORTIVE):
        return "SUPPORTIVE_FEATURE"

    return "NO_TARGET_FEATURE_DETECTED"

df["attention_status"] = df.apply(
    classify_attention, axis=1
)

df["detected_features"] = df.apply(
    lambda row: [
        FEATURE_LABELS[k]
        for k in FEATURES
        if row[k] == 1
    ],
    axis=1
)

df["requires_review"] = df[
    "attention_status"
].isin([
    "POTENTIAL_BARRIER",
    "VERIFY_STEP_FREE_ACCESS"
])

# ------------------------------------------------------------
# 08 — CREATE CLEAN INVENTORY
# ------------------------------------------------------------

output_columns = [
    "image_id",
    "sample_id",
    "side",
    "latitude",
    "longitude",
    "geolocation_status",
    "relative_image_path",
    "image_status",
    "attention_status",
    "requires_review"
] + FEATURES

inventory = df[output_columns].copy()

inventory["detected_features"] = (
    df["detected_features"]
    .apply(json.dumps)
)

inventory["model"] = "qwen2.5vl:7b"
inventory["classification_method"] = "binary_v1"

# ------------------------------------------------------------
# 09 — EXPORT CSV
# ------------------------------------------------------------

CSV_PATH = DATA_OUT / "mobility_inventory.csv"

inventory.to_csv(
    CSV_PATH,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 10 — EXPORT JSON
# ------------------------------------------------------------

json_records = inventory.where(
    pd.notna(inventory), None
).to_dict(orient="records")

# Preserve actual arrays, not JSON strings
for record in json_records:
    record["detected_features"] = json.loads(
        record["detected_features"]
    )

JSON_PATH = DATA_OUT / "mobility_inventory.json"

with open(JSON_PATH, "w", encoding="utf-8") as f:
    json.dump(
        json_records,
        f,
        indent=2,
        ensure_ascii=False
    )

# ------------------------------------------------------------
# 11 — EXPORT GEOJSON
# ------------------------------------------------------------

geolocated = inventory[
    inventory["latitude"].notna()
    & inventory["longitude"].notna()
].copy()

if not geolocated.empty:

    geo = gpd.GeoDataFrame(
        geolocated,
        geometry=gpd.points_from_xy(
            geolocated.longitude,
            geolocated.latitude
        ),
        crs="EPSG:4326"
    )

    geo.to_file(
        DATA_OUT / "mobility_locations.geojson",
        driver="GeoJSON"
    )

# ------------------------------------------------------------
# 12 — FEATURE SUMMARY
# ------------------------------------------------------------

summary = pd.DataFrame({
    "feature": FEATURES,
    "positive_images": [
        int(inventory[k].eq(1).sum())
        for k in FEATURES
    ]
})

summary.to_csv(
    DATA_OUT / "feature_summary.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 13 — VALIDATION REPORT
# ------------------------------------------------------------

validation = inventory[
    [
        "image_id",
        "image_status",
        "geolocation_status",
        "attention_status"
    ]
].copy()

validation.to_csv(
    VALIDATION_OUT / "handover_validation.csv",
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 14 — README
# ------------------------------------------------------------

readme = """# Tiong Bahru–Havelock Mobility Intelligence

Model: Qwen2.5-VL:7B
Classification: Six binary mobility features.

## Data files

- data/mobility_inventory.csv
- data/mobility_inventory.json
- data/mobility_locations.geojson (when coordinates available)
- data/feature_summary.csv

## Images

Photographs are stored in images/.
Use relative_image_path to load each photograph.

## Attention categories

POTENTIAL_BARRIER:
Construction obstruction or stairs detected.

VERIFY_STEP_FREE_ACCESS:
Overhead bridge or underpass detected.

SUPPORTIVE_FEATURE:
Covered walkway or ramp detected.

NO_TARGET_FEATURE_DETECTED:
None of the six target features detected.

## Important limitations

Model predictions are not manually verified.
Feature presence does not prove route obstruction.
Coordinates refer to sampling locations unless
otherwise documented.

Do not interpret these predictions as a
definitive accessibility assessment.
"""

(DOC_OUT / "README.md").write_text(
    readme,
    encoding="utf-8"
)

# ------------------------------------------------------------
# 15 — FINAL REPORT
# ------------------------------------------------------------

print("\n" + "=" * 65)
print("QWEN UI HANDOVER PACKAGE")
print("=" * 65)

print("Images:", len(inventory))

print(
    "Available image files:",
    (inventory["image_status"] == "available").sum()
)

print(
    "Geolocated records:",
    (inventory["geolocation_status"] == "matched").sum()
)

print(
    "Potential barriers:",
    (inventory["attention_status"] == "POTENTIAL_BARRIER").sum()
)

print("\nHandover directory:")
print(HANDOVER_DIR)

print("\nPACKAGE COMPLETE")


In [ ]:

# ============================================================
# CELL 09 — GEOREFERENCED UI HANDOVER
# QWEN 7B + ORIGINAL BLOCK20 STREET VIEW METADATA
# ============================================================

from pathlib import Path
import pandas as pd
import geopandas as gpd
import json

# ------------------------------------------------------------
# 01 — PROJECT PATHS
# ------------------------------------------------------------

BASE = Path(
    r"C:\Users\joshu\OneDrive - National University of Singapore"
    r"\_MArch\_MArch 1\Sem 1"
    r"\AR5958A Artificial Intelligence for Urban Planning"
    r"\Group Assignment\20261108"
)

PROJECT_DIR = BASE / "Qwen_TB"
HANDOVER_DIR = PROJECT_DIR / "UI_Handover"
DATA_DIR = HANDOVER_DIR / "data"
VALIDATION_DIR = HANDOVER_DIR / "validation"

DATA_DIR.mkdir(parents=True, exist_ok=True)
VALIDATION_DIR.mkdir(parents=True, exist_ok=True)

GEO_PATH = (
    BASE / "Incoming"
    / "svi_output_block20"
    / "svi_output_block20"
    / "SVI_block20_LeftRight.geojson"
)

# Alternative location if the GeoJSON is elsewhere
if not GEO_PATH.exists():
    GEO_PATH = (
        BASE / "Incoming"
        / "output_block20"
        / "SVI_block20_LeftRight.geojson"
    )

if not GEO_PATH.exists():
    raise FileNotFoundError(
        "Cannot locate SVI_block20_LeftRight.geojson"
    )

INVENTORY_PATH = DATA_DIR / "mobility_inventory.csv"

if not INVENTORY_PATH.exists():
    raise FileNotFoundError(INVENTORY_PATH)

# ------------------------------------------------------------
# 02 — LOAD EXISTING DATA
# ------------------------------------------------------------

inventory = pd.read_csv(INVENTORY_PATH)

geo = gpd.read_file(GEO_PATH)

if geo.crs is None:
    raise ValueError("GeoJSON has no defined CRS.")

geo = geo.to_crs("EPSG:4326")

print("Qwen image records:", len(inventory))
print("Geographic records:", len(geo))

# ------------------------------------------------------------
# 03 — VALIDATE IMAGE IDENTIFIERS
# ------------------------------------------------------------

inventory["image_id"] = (
    inventory["image_id"].astype(str).str.strip()
)

geo["image_id"] = (
    geo["image_id"].astype(str).str.strip()
)

if inventory["image_id"].duplicated().any():
    raise ValueError("Duplicate Qwen image IDs.")

if geo["image_id"].duplicated().any():
    raise ValueError("Duplicate geographic image IDs.")

# ------------------------------------------------------------
# 04 — EXTRACT GEOGRAPHIC METADATA
# ------------------------------------------------------------

geo["latitude"] = geo.geometry.y
geo["longitude"] = geo.geometry.x

metadata_columns = [
    "image_id",
    "latitude",
    "longitude",
    "road_name",
    "osm_id",
    "fclass",
    "route_heading_deg",
    "camera_heading_deg",
    "camera_pitch_deg",
    "camera_fov_deg",
    "pano_id",
    "pano_lat",
    "pano_lon",
    "match_distance_m",
    "capture_date",
    "image_provider"
]

metadata_columns = [
    c for c in metadata_columns
    if c in geo.columns
]

metadata = pd.DataFrame(
    geo[metadata_columns]
)

# ------------------------------------------------------------
# 05 — JOIN QWEN RESULTS WITH GEOJSON
# ------------------------------------------------------------

# Remove old geographic fields from the previous
# incomplete handover, retaining Qwen predictions.

old_geo_fields = [
    "latitude",
    "longitude",
    "geolocation_status",
    "road_name",
    "osm_id",
    "fclass",
    "route_heading_deg",
    "camera_heading_deg",
    "camera_pitch_deg",
    "camera_fov_deg",
    "pano_id",
    "pano_lat",
    "pano_lon",
    "match_distance_m",
    "capture_date",
    "image_provider"
]

inventory = inventory.drop(
    columns=[
        c for c in old_geo_fields
        if c in inventory.columns
    ]
)

merged = inventory.merge(
    metadata,
    on="image_id",
    how="left",
    validate="one_to_one",
    indicator=True
)

merged["geolocation_status"] = merged["_merge"].map({
    "both": "matched",
    "left_only": "unmatched",
    "right_only": "unmatched"
}).astype(str)

merged = merged.drop(columns="_merge")

# ------------------------------------------------------------
# 06 — ADD UI-FRIENDLY FIELDS
# ------------------------------------------------------------

merged["location_label"] = (
    merged["road_name"].fillna("Unknown road")
    if "road_name" in merged.columns
    else "Unknown road"
)

merged["location_type"] = "sampling_point"

# Paths relative to the UI_Handover root
merged["image_url"] = (
    merged["relative_image_path"]
    .fillna("")
    .astype(str)
    .str.replace("\\", "/", regex=False)
)

# ------------------------------------------------------------
# 07 — SAVE UPDATED CSV
# ------------------------------------------------------------

CSV_OUT = DATA_DIR / "mobility_inventory.csv"

merged.to_csv(
    CSV_OUT,
    index=False,
    encoding="utf-8-sig"
)

# ------------------------------------------------------------
# 08 — SAVE UI JSON
# ------------------------------------------------------------

def clean_json_value(value):
    if isinstance(value, (list, dict)):
        return value
    if pd.isna(value):
        return None
    if hasattr(value, "item"):
        return value.item()
    return value

json_records = []

for record in merged.to_dict(orient="records"):
    clean = {
        k: clean_json_value(v)
        for k, v in record.items()
    }

    features = clean.get("detected_features")

    if isinstance(features, str):
        try:
            clean["detected_features"] = json.loads(features)
        except json.JSONDecodeError:
            clean["detected_features"] = []

    json_records.append(clean)

JSON_OUT = DATA_DIR / "mobility_inventory.json"

with open(JSON_OUT, "w", encoding="utf-8") as f:
    json.dump(
        json_records,
        f,
        ensure_ascii=False,
        indent=2,
        allow_nan=False
    )

# ------------------------------------------------------------
# 09 — SAVE GEOJSON
# ------------------------------------------------------------

valid = merged[
    merged["latitude"].notna()
    & merged["longitude"].notna()
].copy()

if not valid.empty:

    mapped = gpd.GeoDataFrame(
        valid,
        geometry=gpd.points_from_xy(
            valid.longitude,
            valid.latitude
        ),
        crs="EPSG:4326"
    )

    GEO_OUT = DATA_DIR / "mobility_locations.geojson"

    mapped.to_file(
        GEO_OUT,
        driver="GeoJSON"
    )

# ------------------------------------------------------------
# 10 — VALIDATION
# ------------------------------------------------------------

validation = merged[
    [
        "image_id",
        "geolocation_status",
        "image_status",
        "attention_status"
    ]
].copy()

validation.to_csv(
    VALIDATION_DIR / "geographic_join_validation.csv",
    index=False
)

print("\n" + "=" * 65)
print("GEOGRAPHIC HANDOVER REPORT")
print("=" * 65)

print("Qwen images:", len(merged))
print(
    "Geolocated images:",
    (merged.geolocation_status == "matched").sum()
)
print(
    "Unmatched images:",
    (merged.geolocation_status == "unmatched").sum()
)

print("\nOutput directory:", DATA_DIR)

display(
    merged[
        [
            "image_id",
            "road_name",
            "latitude",
            "longitude",
            "attention_status",
            "image_url"
        ]
    ].head(10)
)

print("\nCELL 09 COMPLETE")


In [ ]:

# ============================================================
# CELL 10 — JOURNEY-BASED SVI RETRIEVAL
# BLOCK 20 JALAN MEMBINA -> NOVO VITA CLINIC
# QWEN2.5-VL:7B | GEOREFERENCED IMAGE SEQUENCE
# ============================================================

from pathlib import Path
import geopandas as gpd
import pandas as pd
import numpy as np
import osmnx as ox
import networkx as nx
import folium
import math

from shapely.geometry import LineString, Point
from PIL import Image, ImageOps, ImageDraw, ImageFont
from IPython.display import display

# ------------------------------------------------------------
# 01 — PROJECT PATHS
# ------------------------------------------------------------

BASE = Path(
    r"C:\Users\joshu\OneDrive - National University of Singapore"
    r"\_MArch\_MArch 1\Sem 1"
    r"\AR5958A Artificial Intelligence for Urban Planning"
    r"\Group Assignment\20261108"
)

HANDOVER_DIR = BASE / "Qwen_TB" / "UI_Handover"
DATA_DIR = HANDOVER_DIR / "data"

JOURNEY_DIR = HANDOVER_DIR / "Journey_Demo"
JOURNEY_DIR.mkdir(parents=True, exist_ok=True)

INVENTORY_PATH = DATA_DIR / "mobility_locations.geojson"

if not INVENTORY_PATH.exists():
    raise FileNotFoundError(INVENTORY_PATH)

images = gpd.read_file(INVENTORY_PATH).to_crs("EPSG:4326")

print("Available georeferenced images:", len(images))

# ------------------------------------------------------------
# 02 — DEFINE JOURNEY
# ------------------------------------------------------------

ORIGIN = (
    "20 Jalan Membina, Singapore 164020"
)

DESTINATION = (
    "131 Jalan Bukit Merah, Singapore 160131"
)

# If automatic geocoding fails, enter verified coordinates
# in (latitude, longitude) order.
ORIGIN_LATLON = None
DESTINATION_LATLON = None

def resolve_location(address, manual=None):
    if manual is not None:
        return tuple(manual)
    return ox.geocode(address)

origin_latlon = resolve_location(
    ORIGIN, ORIGIN_LATLON
)

destination_latlon = resolve_location(
    DESTINATION, DESTINATION_LATLON
)

print("Origin:", origin_latlon)
print("Destination:", destination_latlon)

# ------------------------------------------------------------
# 03 — DOWNLOAD WALKING NETWORK
# ------------------------------------------------------------

centre_lat = (
    origin_latlon[0] + destination_latlon[0]
) / 2

centre_lon = (
    origin_latlon[1] + destination_latlon[1]
) / 2

G = ox.graph_from_point(
    (centre_lat, centre_lon),
    dist=1800,
    network_type="walk",
    simplify=True
)

G = ox.project_graph(G, to_crs="EPSG:3414")

origin_point = gpd.GeoSeries(
    [Point(origin_latlon[1], origin_latlon[0])],
    crs="EPSG:4326"
).to_crs("EPSG:3414").iloc[0]

destination_point = gpd.GeoSeries(
    [Point(destination_latlon[1], destination_latlon[0])],
    crs="EPSG:4326"
).to_crs("EPSG:3414").iloc[0]

origin_node = ox.distance.nearest_nodes(
    G, origin_point.x, origin_point.y
)

destination_node = ox.distance.nearest_nodes(
    G, destination_point.x, destination_point.y
)

route_nodes = nx.shortest_path(
    G,
    origin_node,
    destination_node,
    weight="length"
)

# ------------------------------------------------------------
# 04 — CONSTRUCT ROUTE GEOMETRY
# ------------------------------------------------------------

route_segments = []

for u, v in zip(route_nodes[:-1], route_nodes[1:]):

    edge_options = G.get_edge_data(u, v)

    key, edge = min(
        edge_options.items(),
        key=lambda item: item[1].get(
            "length", float("inf")
        )
    )

    start = (
        G.nodes[u]["x"],
        G.nodes[u]["y"]
    )

    end = (
        G.nodes[v]["x"],
        G.nodes[v]["y"]
    )

    geom = edge.get("geometry")

    if geom is None:
        coords = [start, end]
    else:
        coords = list(geom.coords)

        # Orient each segment along walking direction
        if (
            Point(coords[-1]).distance(Point(start))
            <
            Point(coords[0]).distance(Point(start))
        ):
            coords.reverse()

    route_segments.append(coords)

route_coords = []

for coords in route_segments:
    if not route_coords:
        route_coords.extend(coords)
    else:
        route_coords.extend(coords[1:])

route_line = LineString(route_coords)

route_gdf = gpd.GeoDataFrame(
    {
        "journey_id": ["TB_JOURNEY_01"],
        "origin": [ORIGIN],
        "destination": [DESTINATION],
        "route_length_m": [route_line.length]
    },
    geometry=[route_line],
    crs="EPSG:3414"
)

route_gdf.to_crs("EPSG:4326").to_file(
    JOURNEY_DIR / "Journey_Route.geojson",
    driver="GeoJSON"
)

print(
    "Reconstructed route length:",
    round(route_line.length),
    "m"
)

# ------------------------------------------------------------
# 05 — MATCH PHOTOGRAPHS TO ROUTE
# ------------------------------------------------------------

images_m = images.to_crs("EPSG:3414").copy()

MAX_ROUTE_DISTANCE_M = 25

images_m["distance_to_route_m"] = (
    images_m.geometry.distance(route_line)
)

nearby = images_m[
    images_m["distance_to_route_m"]
    <= MAX_ROUTE_DISTANCE_M
].copy()

# Project each image onto route centreline.
# This gives walking distance from route start.
nearby["journey_distance_m"] = nearby.geometry.apply(
    lambda p: route_line.project(p)
)

nearby = nearby.sort_values(
    ["journey_distance_m", "image_id"]
).reset_index(drop=True)

nearby["sequence_no"] = np.arange(
    1, len(nearby) + 1
)

# ------------------------------------------------------------
# 06 — CLASSIFY ACCESSIBILITY ATTENTION
# ------------------------------------------------------------

FEATURES = [
    "covered_walkway",
    "construction_obstruction",
    "stairs",
    "pedestrian_overhead_bridge",
    "ramp",
    "underpass"
]

for feature in FEATURES:
    nearby[feature] = pd.to_numeric(
        nearby[feature],
        errors="coerce"
    ).fillna(0).astype(int)

def classify(row):

    if (
        row["construction_obstruction"] == 1
        or row["stairs"] == 1
    ):
        return "POTENTIAL_BARRIER"

    if (
        row["pedestrian_overhead_bridge"] == 1
        or row["underpass"] == 1
    ):
        return "VERIFY_STEP_FREE_ACCESS"

    if (
        row["covered_walkway"] == 1
        or row["ramp"] == 1
    ):
        return "SUPPORTIVE_FEATURE"

    return "NO_TARGET_FEATURE_DETECTED"

nearby["journey_attention"] = nearby.apply(
    classify, axis=1
)

# ------------------------------------------------------------
# 07 — EXPORT ORDERED IMAGE INVENTORY
# ------------------------------------------------------------

ordered = nearby.to_crs("EPSG:4326").copy()

ordered["latitude"] = ordered.geometry.y
ordered["longitude"] = ordered.geometry.x

ORDERED_CSV = (
    JOURNEY_DIR / "Journey_OrderedImages.csv"
)

ordered.drop(columns="geometry").to_csv(
    ORDERED_CSV,
    index=False,
    encoding="utf-8-sig"
)

ordered.to_file(
    JOURNEY_DIR / "Journey_ImageLocations.geojson",
    driver="GeoJSON"
)

print("Images within route buffer:", len(ordered))

print("\nAttention categories:")
print(ordered["journey_attention"].value_counts())

# ------------------------------------------------------------
# 08 — INTERACTIVE JOURNEY MAP
# ------------------------------------------------------------

m = folium.Map(
    location=[centre_lat, centre_lon],
    zoom_start=16,
    tiles="OpenStreetMap"
)

folium.GeoJson(
    route_gdf.to_crs("EPSG:4326"),
    style_function=lambda feature: {
        "color": "#2457D6",
        "weight": 5
    },
    name="Walking route"
).add_to(m)

COLORS = {
    "POTENTIAL_BARRIER": "#D64545",
    "VERIFY_STEP_FREE_ACCESS": "#E49B37",
    "SUPPORTIVE_FEATURE": "#298B69",
    "NO_TARGET_FEATURE_DETECTED": "#777777"
}

for _, row in ordered.iterrows():

    color = COLORS.get(
        row["journey_attention"],
        "#777777"
    )

    image_link = str(
        row.get("image_url", "")
    )

    popup = (
        f"<b>{row['image_id']}</b><br>"
        f"Sequence: {row['sequence_no']}<br>"
        f"Journey distance: "
        f"{row['journey_distance_m']:.0f} m<br>"
        f"Route offset: "
        f"{row['distance_to_route_m']:.1f} m<br>"
        f"Status: {row['journey_attention']}<br>"
        f"Image: {image_link}"
    )

    folium.CircleMarker(
        location=[row["latitude"], row["longitude"]],
        radius=5,
        color=color,
        fill=True,
        fill_opacity=0.85,
        popup=folium.Popup(popup, max_width=320)
    ).add_to(m)

folium.Marker(
    origin_latlon,
    tooltip="Origin",
    icon=folium.Icon(color="green")
).add_to(m)

folium.Marker(
    destination_latlon,
    tooltip="Destination",
    icon=folium.Icon(color="red")
).add_to(m)

MAP_PATH = JOURNEY_DIR / "Journey_ObstacleMap.html"

m.save(str(MAP_PATH))

display(m)

# ------------------------------------------------------------
# 09 — PHOTOGRAPHIC JOURNEY SEQUENCE
# ------------------------------------------------------------

# Keep a manageable number of images for the poster.
MAX_DISPLAY = 12

if len(ordered) > MAX_DISPLAY:

    indices = np.linspace(
        0,
        len(ordered) - 1,
        MAX_DISPLAY,
        dtype=int
    )

    sequence = ordered.iloc[indices].copy()

else:
    sequence = ordered.copy()

TILE_W = 350
TILE_H = 300
PHOTO_H = 220
COLS = 4

rows = max(
    1,
    math.ceil(len(sequence) / COLS)
)

sheet = Image.new(
    "RGB",
    (COLS * TILE_W, rows * TILE_H + 90),
    "white"
)

draw = ImageDraw.Draw(sheet)
font = ImageFont.load_default()

draw.text(
    (20, 20),
    "TB JOURNEY 01 | ACCESSIBILITY EVIDENCE",
    fill="black",
    font=font
)

for i, (_, row) in enumerate(sequence.iterrows()):

    col = i % COLS
    grid_row = i // COLS

    x = col * TILE_W
    y = 90 + grid_row * TILE_H

    relative_path = str(
        row.get("relative_image_path", "")
    )

    image_path = (
        HANDOVER_DIR / relative_path
    )

    status = row["journey_attention"]
    color = COLORS.get(status, "#777777")

    if image_path.is_file():

        with Image.open(image_path) as im:
            image = ImageOps.exif_transpose(
                im
            ).convert("RGB")

        image.thumbnail(
            (TILE_W - 20, PHOTO_H - 10)
        )

        px = x + (TILE_W - image.width) // 2
        py = y + 8

        sheet.paste(image, (px, py))

    # Attention border
    draw.rectangle(
        (
            x + 5,
            y + 5,
            x + TILE_W - 5,
            y + PHOTO_H + 5
        ),
        outline=color,
        width=5
    )

    draw.text(
        (x + 12, y + PHOTO_H + 15),
        f"{row['image_id']} | "
        f"{row['journey_distance_m']:.0f} m",
        fill="black",
        font=font
    )

    draw.text(
        (x + 12, y + PHOTO_H + 38),
        status.replace("_", " "),
        fill=color,
        font=font
    )

SEQUENCE_PATH = (
    JOURNEY_DIR / "Journey_ImageSequence.png"
)

sheet.save(SEQUENCE_PATH)

display(sheet)

print("\n" + "=" * 65)
print("JOURNEY RETRIEVAL COMPLETE")
print("=" * 65)

print("Route length:", round(route_line.length), "m")
print("Images retrieved:", len(ordered))
print("Sequence images displayed:", len(sequence))
print("Output directory:", JOURNEY_DIR)
